Preprocessing Main Data


In [65]:
# Standard Library
import re

# Third-Party Library
import pandas as pd
import nltk
import contractions
import spacy
from tqdm import tqdm
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.sentiment.util import mark_negation

# NLTK Resources
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

# Load SpaCy Model
nlp = spacy.load(
    "en_core_web_sm",
    disable=["parser", "ner"]
)

In [30]:
# Load the dataset
file_path = "../data/raw/main_data.csv"
df = pd.read_csv(file_path)

print('Available Columns: ', df.columns)
print('First 5 Rows:')
df.head()

Available Columns:  Index(['review', 'sentiment'], dtype='str')
First 5 Rows:


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [31]:
# Normalize

# Normalize Contractions (e.g. she's -> she is)
def normalize_contractions(text):
    return contractions.fix(str(text))

df['normalized'] = df['review'].apply(normalize_contractions)


# Normalize Text
def normalize_text(text):
    # Remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)

    # Lowercase text
    text = text.lower()

    # Remove repeated characters (elongation)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df['normalized'] = df['normalized'].apply(normalize_text)

df[['review', 'normalized']].sample(10, random_state=42)

,review,normalized
33553,I really liked this Summerslam due to the look...,i really liked this summerslam due to the look...
9427,Not many television shows appeal to quite as m...,not many television shows appeal to quite as m...
199,The film quickly gets to a major chase scene w...,the film quickly gets to a major chase scene w...
12447,Jane Austen would definitely approve of this o...,jane austen would definitely approve of this o...
39489,Expectations were somewhat high for me when I ...,expectations were somewhat high for me when i ...
42724,I've watched this movie on a fairly regular ba...,i have watched this movie on a fairly regular ...
10822,For once a story of hope highlighted over the ...,for once a story of hope highlighted over the ...
49498,"Okay, I didn't get the Purgatory thing the fir...","okay, i did not get the purgatory thing the fi..."
4144,I was very disappointed with this series. It h...,i was very disappointed with this series. it h...
36958,The first 30 minutes of Tinseltown had my fing...,the first 30 minutes of tinseltown had my fing...


In [38]:
# Tokenization

df['tokens'] = df['normalized'].apply(lambda x: word_tokenize(x))

df[['normalized', 'tokens']].head(5)


,normalized,tokens
0,one of the other reviewers has mentioned that ...,"[one, of, the, other, reviewers, has, mentione..."
1,a wonderful little production. the filming tec...,"[a, wonderful, little, production, ., the, fil..."
2,i thought this was a wonderful way to spend ti...,"[i, thought, this, was, a, wonderful, way, to,..."
3,basically there is a family where a little boy...,"[basically, there, is, a, family, where, a, li..."
4,"petter mattei's ""love in the time of money"" is...","[petter, mattei, 's, ``, love, in, the, time, ..."


In [45]:
# Standardization Text (Lemmatization)

def lemmatize_tokens(tokens):
    text = ' '.join(tokens)
    doc = nlp(text)

    return [token.lemma_ for token in doc]

df['processed_tokens'] = df['tokens'].apply(lemmatize_tokens)

df[['tokens', 'processed_tokens']].head(10)

,tokens,processed_tokens
0,"[one, of, the, other, reviewers, has, mentione...","[one, of, the, other, reviewer, have, mention,..."
1,"[a, wonderful, little, production, ., the, fil...","[a, wonderful, little, production, ., the, fil..."
2,"[i, thought, this, was, a, wonderful, way, to,...","[I, think, this, be, a, wonderful, way, to, sp..."
3,"[basically, there, is, a, family, where, a, li...","[basically, there, be, a, family, where, a, li..."
4,"[petter, mattei, 's, ``, love, in, the, time, ...","[petter, mattei, 's, `, `, love, in, the, time..."
5,"[probably, my, all-time, favorite, movie, ,, a...","[probably, my, all, -, time, favorite, movie, ..."
6,"[i, sure, would, like, to, see, a, resurrectio...","[I, sure, would, like, to, see, a, resurrectio..."
7,"[this, show, was, an, amazing, ,, fresh, &, in...","[this, show, be, an, amazing, ,, fresh, &, inn..."
8,"[encouraged, by, the, positive, comments, abou...","[encourage, by, the, positive, comment, about,..."
9,"[if, you, like, original, gut, wrenching, laug...","[if, you, like, original, gut, wrench, laughte..."


In [46]:
# Negation Handling

df['tokens_negated'] = df['processed_tokens'].apply(mark_negation)

df[['tokens', 'processed_tokens', 'tokens_negated']].head(10)

,tokens,processed_tokens,tokens_negated
0,"[one, of, the, other, reviewers, has, mentione...","[one, of, the, other, reviewer, have, mention,...","[one, of, the, other, reviewer, have, mention,..."
1,"[a, wonderful, little, production, ., the, fil...","[a, wonderful, little, production, ., the, fil...","[a, wonderful, little, production, ., the, fil..."
2,"[i, thought, this, was, a, wonderful, way, to,...","[I, think, this, be, a, wonderful, way, to, sp...","[I, think, this, be, a, wonderful, way, to, sp..."
3,"[basically, there, is, a, family, where, a, li...","[basically, there, be, a, family, where, a, li...","[basically, there, be, a, family, where, a, li..."
4,"[petter, mattei, 's, ``, love, in, the, time, ...","[petter, mattei, 's, `, `, love, in, the, time...","[petter, mattei, 's, `, `, love, in, the, time..."
5,"[probably, my, all-time, favorite, movie, ,, a...","[probably, my, all, -, time, favorite, movie, ...","[probably, my, all, -, time, favorite, movie, ..."
6,"[i, sure, would, like, to, see, a, resurrectio...","[I, sure, would, like, to, see, a, resurrectio...","[I, sure, would, like, to, see, a, resurrectio..."
7,"[this, show, was, an, amazing, ,, fresh, &, in...","[this, show, be, an, amazing, ,, fresh, &, inn...","[this, show, be, an, amazing, ,, fresh, &, inn..."
8,"[encouraged, by, the, positive, comments, abou...","[encourage, by, the, positive, comment, about,...","[encourage, by, the, positive, comment, about,..."
9,"[if, you, like, original, gut, wrenching, laug...","[if, you, like, original, gut, wrench, laughte...","[if, you, like, original, gut, wrench, laughte..."


In [55]:
# Stopwords Removal

stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    cleaned = []

    for word in tokens:

        # Remove suffix _NEG
        base_word = word.removesuffix('_NEG')

        # Normalize for checking
        check_word = base_word.lower()

        # Remove punctuation / symbol-only token
        if not any(char.isalnum() for char in base_word):
            continue

        # Remove possessive
        if check_word in {"'s", "’s"}:
            continue

        # Remove stopword
        if check_word in stop_words:
            continue

        # Keep the original token, including *_NEG and numbers
        cleaned.append(word)

    return cleaned

df['no_stopwords'] = df['tokens_negated'].apply(remove_stopwords)

df[df['tokens_negated'].apply(
    lambda tokens: any(word.endswith('_NEG') for word in tokens)
)][[
    'processed_tokens',
    'tokens_negated',
    'no_stopwords'
]].head(10)


,processed_tokens,tokens_negated,no_stopwords
0,"[one, of, the, other, reviewer, have, mention,...","[one, of, the, other, reviewer, have, mention,...","[one, reviewer, mention, watch, 1, oz, episode..."
1,"[a, wonderful, little, production, ., the, fil...","[a, wonderful, little, production, ., the, fil...","[wonderful, little, production, filming, techn..."
2,"[I, think, this, be, a, wonderful, way, to, sp...","[I, think, this, be, a, wonderful, way, to, sp...","[think, wonderful, way, spend, time, hot, summ..."
4,"[petter, mattei, 's, `, `, love, in, the, time...","[petter, mattei, 's, `, `, love, in, the, time...","[petter, mattei, love, time, money, visually, ..."
5,"[probably, my, all, -, time, favorite, movie, ...","[probably, my, all, -, time, favorite, movie, ...","[probably, time, favorite, movie, story, selfl..."
7,"[this, show, be, an, amazing, ,, fresh, &, inn...","[this, show, be, an, amazing, ,, fresh, &, inn...","[show, amazing, fresh, innovative, idea, 70, f..."
8,"[encourage, by, the, positive, comment, about,...","[encourage, by, the, positive, comment, about,...","[encourage, positive, comment, film, look, for..."
10,"[phil, the, alien, be, one, of, those, quirky,...","[phil, the, alien, be, one, of, those, quirky,...","[phil, alien, one, quirky, film, humour, base,..."
11,"[I, see, this, movie, when, I, be, about, 12, ...","[I, see, this, movie, when, I, be, about, 12, ...","[see, movie, 12, come, recall, scary, scene, b..."
12,"[so, I, be, not, a, big, fan, of, boll, 's, wo...","[so, I, be, not, a_NEG, big_NEG, fan_NEG, of_N...","[big_NEG, fan_NEG, boll_NEG, work_NEG, many_NE..."


In [58]:
# Generating Processed Text
# For TF-IDF Vectorization

df['processed_text'] = df['no_stopwords'].apply(
    lambda tokens: ' '.join(tokens)
)

df[['no_stopwords','processed_text']].head()

,no_stopwords,processed_text
0,"[one, reviewer, mention, watch, 1, oz, episode...",one reviewer mention watch 1 oz episode hook r...
1,"[wonderful, little, production, filming, techn...",wonderful little production filming technique ...
2,"[think, wonderful, way, spend, time, hot, summ...",think wonderful way spend time hot summer week...
3,"[basically, family, little, boy, jake, think, ...",basically family little boy jake think zombie ...
4,"[petter, mattei, love, time, money, visually, ...",petter mattei love time money visually stunnin...


In [61]:
# Generate Dataframe into [id_data, sentiment_label, review, processed_text, processed_tokens]
df_generated = pd.DataFrame({
    'id_data': range(1, len(df) + 1),
    'sentiment_label': df['sentiment'],
    'review': df['review'],
    'processed_text': df['processed_text'],
    'processed_tokens': df['no_stopwords'],
})

df_generated.info()
df_generated.shape
df_generated.head()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id_data           50000 non-null  int64 
 1   sentiment_label   50000 non-null  str   
 2   review            50000 non-null  str   
 3   processed_text    50000 non-null  str   
 4   processed_tokens  50000 non-null  object
dtypes: int64(1), object(1), str(3)
memory usage: 1.9+ MB


,id_data,sentiment_label,review,processed_text,processed_tokens
0,1,positive,One of the other reviewers has mentioned that ...,one reviewer mention watch 1 oz episode hook r...,"[one, reviewer, mention, watch, 1, oz, episode..."
1,2,positive,A wonderful little production. <br /><br />The...,wonderful little production filming technique ...,"[wonderful, little, production, filming, techn..."
2,3,positive,I thought this was a wonderful way to spend ti...,think wonderful way spend time hot summer week...,"[think, wonderful, way, spend, time, hot, summ..."
3,4,negative,Basically there's a family where a little boy ...,basically family little boy jake think zombie ...,"[basically, family, little, boy, jake, think, ..."
4,5,positive,"Petter Mattei's ""Love in the Time of Money"" is...",petter mattei love time money visually stunnin...,"[petter, mattei, love, time, money, visually, ..."


In [66]:
# Export Preprocessed Data to CSV
output_file_path = '../data/preprocessed_data.csv'
chunk_size = 5000
total_rows = len(df_generated)

# Write header to CSV file first
df_generated.iloc[0:0].to_csv(output_file_path, index=False)

# Looping with tqdm bar
for start in tqdm(range(0, total_rows, chunk_size), desc="Exporting Progress"):
    end = min(start + chunk_size, total_rows)
    chunk = df_generated.iloc[start:end]
    
    # Append chunk to CSV file without header
    chunk.to_csv(output_file_path, mode='a', index=False, header=False)

print(f"\nSuccessfully exported to: {output_file_path}")

Exporting Progress: 100%|██████████| 10/10 [00:06<00:00,  1.66it/s]


Successfully exported to: ../data/preprocessed_data.csv
